# Analysis of action potentials with the IPFX library

In this tutorial, I explain how to analyze action potentials using the [IPFX library](https://github.com/AllenInstitute/ipfx) created by the Allen Institute for Brain Science. Use and credit the library according to its [license](https://github.com/AllenInstitute/ipfx?tab=License-1-ov-file).

To read the full tutorial, please visit [Patch-clamp data analysis in Python: action potentials](https://spikesandbursts.wordpress.com/2022/05/03/patch-clamp-analysis-python-action-potentials/) of the [Spikes and Bursts](https://spikesandbursts.wordpress.com/) blog.

**References**
- [IPFX documentation](https://ipfx.readthedocs.io/en/latest/)

# Import the libraries

**Important**: IPFX generally requires a separate environment with Python version <3.10. 

In [ ]:
# Import the packages 
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# To import pClamp abf files
import pyabf

# IPFX library
from ipfx.feature_extractor import (SpikeFeatureExtractor,
                                    SpikeTrainFeatureExtractor)

# Backend for interactive plots (optional)
# %matplotlib widget  
# plt.close('all')

# Create the paths

In [ ]:
notebook_name = 'action_potentials_ipfx'

# Data path to 'Data_example' folders. Change accordingly to your data structure.
data_path = os.path.dirname(os.getcwd())  # Moves one level up from the current directory

# Change the folder names accordingly
paths = {'data':  f'{data_path}/Data',
         'processed_data': f'{data_path}/Processed_data/{notebook_name}',
         'analysis': f'{data_path}/Analysis/{notebook_name}'}

# Make folders if they do not exist yet
for path in paths.values():
    os.makedirs(path, exist_ok=True)

# Load the example data

Example data for this notebook in GitHub's data folder:
* ABF files: **pfc_pvalb_aps_01.abf**, **pfc_pvalb_aps_02.abf**
* CSV file: **pfc_pvalb_aps_01.csv**

In [ ]:
# ABF file
filename = "pfc_pvalb_aps_01"

data_path = f"{paths['data']}/{filename}.abf" 
abf = pyabf.ABF(data_path)
print(abf)

# Plot the traces

## Quick plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)

voltage_channel = 0
current_channel = 0

for sweep in abf.sweepList:
    abf.setSweep(sweep, voltage_channel)
    ax1.plot(abf.sweepX, abf.sweepY, alpha=0.5)
    ax1.set_ylabel(abf.sweepLabelY)

    abf.setSweep(sweep, channel=current_channel)
    ax2.plot(abf.sweepX, abf.sweepC, color='black')
    ax2.set_ylabel(abf.sweepLabelC)
    ax2.set_xlabel(abf.sweepLabelX)
    ax2.set_xlim(0, 1)

plt.show()

## Customize plot

In [ ]:
# Font type and size
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 14

# Figure size and format
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 4), 
                               gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

# Select channels and sweeps
voltage_channel = 0
current_channel = 0
sweep_number = 11

# Assuming `abf` is already loaded and prepared.
for sweep in abf.sweepList:

    # Plotting voltage
    abf.setSweep(sweep_number, voltage_channel)
    ax1.plot(abf.sweepX, abf.sweepY, color='red', linewidth=0.5)
    ax1.set_ylabel("V (mV)")

    # Customize ax1
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.spines['bottom'].set_visible(False)
    ax1.spines['left'].set_position(('outward', 10))
    ax1.tick_params(bottom=False)  # hide ticks

    # Plotting current
    abf.setSweep(sweep_number, current_channel)
    ax2.plot(abf.sweepX, abf.sweepC, color='black', linewidth=0.5)
    ax2.set_ylabel("I (pA)")
    ax2.set_xlabel("Time (s)")
    ax2.set_xlim(0.2, 0.8)   # visible x-range

    # Customize spines for a floating effect
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.spines['bottom'].set_visible(True)  # ensure bottom spine is visible
    ax2.spines['bottom'].set_position(('outward', 10))
    ax2.spines['left'].set_position(('outward', 10))

plt.tight_layout()
plt.show()

# Save the plot and the table
fig.savefig(f"{paths['analysis']}/{filename}_sweep{sweep_number}.svg", dpi=300)

# Spike feafures: one or more traces
* You can adapt the range of the analysis to one action potentrial or longer current clamp recordings by adjusting: `start, end` (in seconds).
* The loop function (for i in x) also works if you have only one trace.

In [ ]:
# Define the analysis window (in seconds)
window_start = 0.24
window_end = 0.265

# Analysis parameters for action potential detection (use the ones you need)
voltage_threshold = 0  # Minimum absolute peak level in mV for spike detection
dv_cutoff = 20.0  # Minimum dV/dt to qualify as a spike in V/s 
min_height = 2  # Minimum acceptable height from threshold to peak in mV
thresh_frac = 0.05  # fraction of average upstroke for threshold calculation
max_interval = 0.005  # Maximum time between start of spike and time of peak in sec (default: 5 ms)
filter = None  # None or cutoff frequency (kHz,default: 10.0) for 4-pole low-pass Bessel filter 

# Set the table from the dataframe below
dataframe = [] 

# Loop function to analyze each voltage trace of the file
for sweepNumber in abf.sweepList:
    abf.setSweep(sweepNumber)
    time = abf.sweepX
    voltage = abf.sweepY
    current = abf.sweepC
    
    # Define the region
    start, end = window_start, window_end
    
    # Parameters for analysis: 
    sfx = SpikeFeatureExtractor (start, end, 
                                 filter=filter, 
                                 dv_cutoff=dv_cutoff,  
                                 thresh_frac=thresh_frac,  
                                 min_peak=voltage_threshold)  
    sfx_results = sfx.process(time, voltage, current)
    dataframe.append(sfx_results)  # To get the mean: df.append(sfx_results.mean())
    
# Table with the features from all the action potentials
table = pd.concat(dataframe)

# Plot the trace/s
plt.figure(figsize=(6,4))
plt.xlabel ("Time (s)")
plt.ylabel("Voltage (mV)")
for sweepNumber in abf.sweepList:  # Loop to plot all the traces
    abf.setSweep(sweepNumber)
    plt.plot(abf.sweepX, abf.sweepY, alpha=.6, label="sweep %d" % (sweepNumber))

    # To highlight one trace
abf.setSweep(3) 
plt.plot(abf.sweepX, abf.sweepY, linewidth=1, color='black')
plt.xlim(window_start, window_end)

# Save the table and plot
plt.savefig(f"{paths['analysis']}/{filename}_spikefeatures_ipfx.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_spikefeatures_ipfx.csv", index=False)
   
# Display the graph and the table
plt.show()
table

# Remove the below # to show only selected columns. E.g:
# columns = ['threshold_i', 'threshold_v', 'width', 'upstroke', 'downstroke']
# table[columns]

# Spike train features: one trace

**Important**: IPFX calculates the firing rate (spikes/sec) instead of the number of action potentials. To get the number of action potentials, either select a 1-second analysis window or multiply the 'avg_rate' by the time window width (e.g., 'avg_rate' * 0.6 s).

In [ ]:
# Select the sweep/trace and channel
sweep_number=10
abf.setSweep(sweepNumber=sweep_number, channel=0)

# Define the analysis window (in seconds)
window_start = 0
window_end = 1

# Analysis parameters
voltage_threshold = 0  # Minimum peak level (mV) for detection
filter = None  # None or cutoff frequency (kHz,default: 10.0) for 4-pole low-pass Bessel filter 
dv_cutoff = 20.0  # Minimum dV/dt to qualify as a spike in V/s
thresh_frac = 0.05  # Fraction of average upstroke for threshold calculation

# Define each variable
time = abf.sweepX
voltage = abf.sweepY
current = abf.sweepC

# Current steps
currents = []  # Current value between t1 and t2 (ms) for each step
t1 = int(400*abf.dataPointsPerMs) 
t2 = int(500*abf.dataPointsPerMs)
current_mean = np.average(abf.sweepC[t1:t2])

# Define the region (in sec) and detection parameters
start, end = window_start, window_end
sfx = SpikeFeatureExtractor (start=start, end=end, 
                             filter=filter,  
                             dv_cutoff=dv_cutoff,  
                             thresh_frac=thresh_frac, 
                             min_peak=voltage_threshold)   
sfx_results = sfx.process(time, voltage, current)
stfx = SpikeTrainFeatureExtractor(start, end)
stfx_results = stfx.process(time, voltage, current, sfx_results)

# Optional: Create a table with the stfx results
table = pd.DataFrame()
length = len(table)
table.loc[length, 'Current_step'] = current_mean
table.loc[length, 'adaptation'] = stfx_results ["adapt"]
table.loc[length, 'Latency'] = stfx_results ["latency"]
table.loc[length, 'ISI_CV'] = stfx_results ["isi_cv"]
table.loc[length, 'ISI_mean'] = stfx_results ["mean_isi"]
table.loc[length, 'ISI_first'] = stfx_results ["first_isi"]
table.loc[length, 'Firing_rate_Hz'] = stfx_results ["avg_rate"]

# Plotting the voltage trace and the current step
fig = plt.figure(figsize=(7, 5)) #Figure size

# Voltage traces
ax1 = fig.add_subplot(211)
ax1.plot(abf.sweepX, abf.sweepY)
# Show detection of action potential peak
ax1.plot(sfx_results["peak_t"], sfx_results["peak_v"], 'r.')
# Show detection of action potential threshold
ax1.plot(sfx_results["threshold_t"], sfx_results["threshold_v"], 'k.')
ax1.set_ylabel("Voltage (mV)")

# Current stimulus
ax2 = fig.add_subplot(212, sharex=ax1) 
ax2.plot(abf.sweepX, abf.sweepC, color='r')
ax2.set_ylabel("Current (pA)")
ax2.set_xlabel("Time (s)")

# Zoom in and out (not necessary if you use %matplotlib widget)
ax2.axes.set_xlim(window_start, window_end) # Shared x-axis (range in sec)

# Save the table and plot
plt.savefig(f"{paths['analysis']}/{filename}_sweep{sweep_number}_spiketrainfeatures_ipfx.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_sweep{sweep_number}_spiketrainfeatures_ipfx.csv", index=False)

# View the graph and the table
plt.show()
table

# Spike train features: several traces

In [ ]:
# Define the analysis window in seconds
window_start = 0.1
window_end = 1.1
highlighted_trace = 3

# Define the region detection parameters
voltage_threshold = 0  # Minimum acceptable absolute peak level in mV 
dv_cutoff = 20.0  # Minimum dV/dt to qualify as a spike in V/s
thresh_frac = 0.05  # Fraction of average upstroke for threshold calculation

# Define the current average segment in ms
current_t1 = 400 
current_t2 = 500

# Define the channels (0 by default)
voltage_channel = 0
current_channel = 0

# Set the dataframe for the table
table = pd.DataFrame()

# Loop function to analyze each voltage and current trace of the file
for sweep in abf.sweepList:
    abf.setSweep(sweep, channel=voltage_channel)
    time = abf.sweepX
    voltage = abf.sweepY
    
    abf.setSweep(sweep, channel=current_channel)
    current = abf.sweepC
    
    # Current value between t1 and t2 (ms) for each step
    currents = []
    t1 = int(current_t1*abf.dataPointsPerMs) 
    t2 = int(current_t2*abf.dataPointsPerMs)
    current_mean = np.average(current[t1:t2])
    
    # Define the region (in sec) and detection parameters
    voltage_threshold = voltage_threshold
    dv_cutoff = dv_cutoff 
    thresh_frac = thresh_frac 
    start, end = window_start, window_end
    
    # Run the function to extract spike features
    sfx = SpikeFeatureExtractor (start=start, end=end, 
                                 filter=None,  
                                 dv_cutoff=dv_cutoff,
                                 thresh_frac=thresh_frac, 
                                 min_peak=voltage_threshold) 
    
    sfx_results = sfx.process(time, voltage, current)
    stfx = SpikeTrainFeatureExtractor(start, end)
    stfx_results = stfx.process(time, voltage, current, sfx_results)
    
    # Create a table with the stfx results
    length = len(table)
    table.loc[length, 'current_step'] = current_mean
    table.loc[length, 'avg_rate'] = stfx_results["avg_rate"] 
    if stfx_results ['avg_rate'] > 0:
        table.loc[length, 'adaptation'] = stfx_results ["adapt"]
        table.loc[length, 'Latency_s'] = stfx_results ["latency"]
        table.loc[length, 'ISI_CV'] = stfx_results ["isi_cv"]
        table.loc[length, 'ISI_mean_ms'] = stfx_results ["mean_isi"]*1000
        table.loc[length, 'ISI_first_ms'] = stfx_results ["first_isi"]*1000
     
# Example with three subplots
fig = plt.figure(figsize=(15, 5))

# Input-output curve
ax1 = fig.add_subplot(1, 2, 1)
currents = []
for sweep in abf.sweepList:
    abf.setSweep(sweep, channel=current_channel)
    currents.append(np.average(abf.sweepC[t1:t2]))
ax1.scatter(currents, table.loc[:,'avg_rate'])
ax1.set_xlabel('Current step (pA)')
ax1.set_ylabel('Action potentials')

# All voltage traces
ax2 = fig.add_subplot(2, 2, 2)
for sweep in abf.sweepList:
    abf.setSweep(sweep, channel=voltage_channel)
    ax2.plot(abf.sweepX, abf.sweepY, alpha=.6, label="Sweep %d" % (sweep))
ax2.set_ylabel('Membrane voltage (mV)')
# ax2.legend() # Optional

# Plot the voltage threshold and time window lines
ax2.axhline(voltage_threshold, color='gray', linestyle='--')
plt.axvline(start, linestyle="dotted", color="gray")
plt.axvline(end, linestyle="dotted", color="gray")

# To highlight one voltage trace
abf.setSweep(highlighted_trace, channel=voltage_channel) 
ax2.plot(abf.sweepX, abf.sweepY, linewidth=1, color='black')
ax2.axes.set_xlim(window_start-0.1, window_end+0.1) # Range of the x-axis (seconds)

# Plot the current steps
ax3 = fig.add_subplot(2, 2, 4, sharex=ax2) 
for sweep in abf.sweepList:
    abf.setSweep(sweep, channel=current_channel)
    current = abf.sweepC
    ax3.plot(abf.sweepX, current)
ax3.set_ylabel("Current (pA)")
ax3.set_xlabel("Time (s)")

# To highlight one current trace
abf.setSweep(highlighted_trace, channel=current_channel) 
ax3.plot(abf.sweepX, abf.sweepC, linewidth=1, color='black')
fig.tight_layout()

# Calculate the rheobase
rheobase_index = table[table['avg_rate'] > 0].index[0]
rheobase = table.loc[rheobase_index, 'current_step']
print("Rheobase (pA):", rheobase)

# Save the table and plot
plt.savefig(f"{paths['analysis']}/{filename}_spiketrainfeatures_ipfx.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_spiketrainfeatures_ipfx.csv", index=False)

# Show the graph and the table results 
plt.show()
table

## Spike train features: text files

In [ ]:
# Load the file
filename = "pfc_pvalb_aps_01"
data_path = f"{paths['data']}/{filename}.csv" 
data = np.loadtxt(fname=data_path, delimiter = ",")

# Create a table
table = pd.DataFrame()

# Define the time, voltage, and current columns
time = data [:, 0]/1000
voltages = data [:, 1:14]
currents = data [:, 14:28]
    
for i, voltage in enumerate(voltages.T):
    current = currents.T[i]
    start, end = 0.1, 1.1
    stfx = SpikeTrainFeatureExtractor (start=start, end=end)
    sfx = SpikeFeatureExtractor(start=start, end=end, filter=None)
    spikes_df = sfx.process(t=time, v=voltage, i=current)
    stfx = stfx.process(t=time, v=voltage, i=current, spikes_df=spikes_df)

    # Table
    length = len(table)
    table.loc[length, 'avg_rate'] = stfx["avg_rate"]
    if stfx['avg_rate'] > 0:
        table.loc[length, 'adaptation'] = stfx ["adapt"]
        table.loc[length, 'Latency'] = stfx ["latency"]
        table.loc[length, 'ISI_CV'] = stfx ["isi_cv"]
        table.loc[length, 'ISI_mean'] = stfx ["mean_isi"]
        table.loc[length, 'ISI_first'] = stfx ["first_isi"]
    
    plt.plot (time, voltage)
    trace_highlight = data[:, 4]  
    plt.plot (time, trace_highlight,  linewidth=1, color='black')
    plt.xlim(0, 1)
    plt.xlabel ("Time (s)")
    plt.ylabel("Voltage (mV)")
    
# Get the current step values
t1 = currents[(time > 0.3) & (time < 0.4), :]
t2 = currents[(time > 0) & (time < 0.2), :]
current_mean = np.median(t1, axis=0) - np.median(t2,axis=0)

# Add the current step values as a new column to the DataFrame 'table'
table['current_steps'] = np.round(current_mean, 0)

# Save the table and plot
plt.savefig(f"{paths['analysis']}/{filename}_csv_spiketrainfeatures_ipfx.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_csv_spiketrainfeatures_ipfx.csv", index=False)

plt.show()
table

# Spike train features: several recordings

## Option A: loop through recordings

In [ ]:
# Set the dataframe for the table
table_all = pd.DataFrame()

experiment = 'pfc_pvalb_aps' 

for file in os.listdir(paths['data']):
    # Filter files by name and extension
    if file.startswith(experiment) and file.endswith('.abf'):
        abf_path = os.path.join(paths['data'], file)
        abf = pyabf.ABF(abf_path)

        # Get the filename without the extension
        filename = os.path.splitext(file)[0]
        
        # Paste below the code you want to apply to the files
        for sweep in abf.sweepList:
            abf.setSweep(sweep)
            time = abf.sweepX
            voltage = abf.sweepY
            current = abf.sweepC

            currents = [] # Current value between t1 and t2 (ms) for each step
            t1 = int(400*abf.dataPointsPerMs) 
            t2 = int(500*abf.dataPointsPerMs)
            current_mean = np.average(abf.sweepC[t1:t2])

            start, end = 0.1, 1.1
            sfx = SpikeFeatureExtractor(start=start, end=end, filter=None)
            sfx_results = sfx.process(time, voltage, current)
            stfx = SpikeTrainFeatureExtractor (start=start, end=end)
            stfx_results = stfx.process(time, voltage, current, sfx_results)

            # Create a table with the stfx results
            length = len(table_all)
            table_all.loc[length, 'file_name'] = filename
            table_all.loc[length, 'current_step'] = current_mean
            table_all.loc[length, 'avg_rate'] = stfx_results["avg_rate"]
            if stfx_results ['avg_rate'] > 0:
                table_all.loc[length, 'adaptation'] = stfx_results ["adapt"]
                table_all.loc[length, 'Latency_s'] = stfx_results ["latency"]
                table_all.loc[length, 'ISI_CV'] = stfx_results ["isi_cv"]
                table_all.loc[length, 'ISI_mean_ms'] = stfx_results ["mean_isi"]*1000
                table_all.loc[length, 'ISI_first_ms'] = stfx_results ["first_isi"]*1000

# Save the table
table.to_csv(f"{paths['analysis']}/{experiment}_all_spiketrainfeatures_ipfx.csv", index=False)

# Show the table (optional)
table_all

## Option B: merge table from single recordings

In [ ]:
# Path for the merged summary file
summary_file = f"{paths['analysis']}/{experiment}_merged_spiketrainfeatures_ipfx.csv"

# Load existing merged table if it exists, otherwise create empty
if os.path.isfile(summary_file):
    table_merge = pd.read_csv(summary_file)
else:
    table_merge = pd.DataFrame()

# Loop through CSV files in the folder
for file in os.listdir(paths['analysis']):
    
    # Adapt the if statement according to your data structure
    if (file.startswith('pfc_pvalb_aps') and  
        file.endswith('spiketrainfeatures_ipfx.csv') and
        'merged' not in file and 'all' not in file and 'sweep' not in file and 'csv' not in os.path.splitext(file)[0]): 
        
        # Skip if file already added
        if 'filename' in table_merge.columns and file in table_merge['filename'].tolist():
            continue

        csv_path = os.path.join(paths['analysis'], file)  # Use the same folder
        table = pd.read_csv(csv_path)
        
        # Add a column with the filename
        table['filename'] = file
        
        # Append to the merged table
        table_merge = pd.concat([table_merge, table], ignore_index=True)

# Save the merged table
table_merge.to_csv(summary_file, index=False, na_rep='NaN')

table_merge